### 构建智能咨询面试官系统指南

要利用您现有的语料库训练一个带有语音功能的智能咨询面试官（Consulting Interviewer），通常需要以下几个核心模块的配合：

#### 1. 数据准备 (Data Preparation)
您已经有了现成的语料（咨询案例、面试对话等）。您需要将这些数据格式化为大语言模型（LLM）微调所需的格式，通常是 `JSONL` 格式的问答对或对话上下文。
例如：
```json
{"instruction": "作为麦肯锡面试官，给候选人出一个关于市场进入策略的Case。", "input": "", "output": "好的，我们今天的Case是关于一家欧洲的传统汽车制造商打算进入中国的新能源汽车市场..."}
```

#### 2. 大语言模型微调 (LLM Fine-tuning)
由于您需要专业的“咨询面试官”人设和逻辑，建议使用开源大模型（如 Qwen, Llama 3, GLM-4 等）进行 **LoRA / QLoRA 微调**。这将极大地降低硬件要求，同时让模型学习您的专业语料。
* **推荐框架**: `HuggingFace Transformers`, `PEFT`, 或者使用现成的微调工具箱如 `LLaMA-Factory`。

#### 3. 语音模块 (Voice Integration)
要实现语音交互，您需要两个组件：
* **ASR (语音转文本)**: 接收候选人的回答。推荐使用 OpenAI 的 **Whisper** 模型，开源且准确率极高。
* **TTS (文本转语音)**: 将面试官的回复转为语音。可以使用 **Edge-TTS**（免费）、**CosyVoice**（支持克隆高保真音色）、或者商业 API（如 ElevenLabs）来获得专业且逼真的面试官声音。

#### 4. 系统集成 (System Integration)
使用 Python 编写一个主循环，或者使用 `LangChain` / `AutoGen` 来管理对话状态。流程如下：
`麦克风收音 -> Whisper (ASR) -> 微调后的 LLM 处理并生成回复 -> TTS 合成语音 -> 播放给候选人`

In [ ]:
# 1. 基础环境配置 (Environment Setup)
# 这里我们安装一些微调和语音处理常用的基础库
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q openai-whisper edge-tts soundfile

print("基础环境安装完成！您可以开始下一步的数据处理了。")

基础环境安装完成！您可以开始下一步的数据处理了。


### 1.5 Efficient Fine-Tuning Setup with Unsloth

[Unsloth](https://github.com/unslothai/unsloth) is a wildly popular library that optimizes Hugging Face models to train significantly faster while using much less GPU memory. It's ideal for Google Colab.

Let's install Unsloth and its specific dependencies.

In [ ]:
# Install Unsloth and its required dependencies for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

print("Unsloth successfully installed! We are ready for high-speed fine-tuning.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0bvoy4fi/unsloth_4ab46db093f74510ac1be38eaa694029
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0bvoy4fi/unsloth_4ab46db093f74510ac1be38eaa694029
  Resolved https://github.com/unslothai/unsloth.git to commit 3044401c2574e1afa2224ac0d1eceb4911881c84
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.2 MB/s eta 0:00:00
   ━━

### 2. 连接 Google Drive (Mount Google Drive)

为了读取您的训练语料并保存微调后的模型，我们需要连接到您的 Google Drive。请在运行下方代码后，点击弹出的窗口允许访问。

In [ ]:
from google.colab import drive

# 挂载 Google Drive 到 /content/drive
drive.mount('/content/drive')

print("\nGoogle Drive 挂载成功！")
print("您的默认云盘路径是: /content/drive/MyDrive/")
print("如果您创建了名为 'Consulting_Data' 的文件夹，可以通过路径 '/content/drive/MyDrive/Consulting_Data' 来访问您的文件。")

MessageError: Error: credential propagation was unsuccessful

### 3. 验证云盘并准备数据目录
检查云盘是否连接成功，并为您预备好存放语料的文件夹。

In [ ]:
# 一键自动安装所有依赖库
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q openai-whisper edge-tts soundfile
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q PyPDF2 python-docx

print("\n✅ 所有依赖库已自动下载并安装完成！您可以继续运行后续的代码了。")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6l16vigk/unsloth_b490765344054de79b08d613c69122cc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6l16vigk/unsloth_b490765344054de79b08d613c69122cc
  Resolved https://github.com/unslothai/unsloth.git to commit 3044401c2574e1afa2224ac0d1eceb4911881c84
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import shutil

# 定义您的云盘根目录
drive_path = '/content/drive/MyDrive/'

if os.path.exists(drive_path):
    print("✅ 恭喜！检测到 Google Drive 已经挂载成功！")

    # 设定主文件夹和原始语料文件夹路径
    data_folder = os.path.join(drive_path, 'Colab Notebooks', 'ow')
    raw_data_folder = os.path.join(data_folder, 'raw_materials')

    # 自动创建文件夹
    os.makedirs(raw_data_folder, exist_ok=True)
    print(f"\n📁 主文件夹: {data_folder}")
    print(f"📁 原始语料子文件夹已创建: {raw_data_folder}")

    # 自动把外层已经上传的语料文件移动到 raw_materials 子文件夹中整理好
    print("\n整理文件中...")
    for filename in os.listdir(data_folder):
        file_path = os.path.join(data_folder, filename)
        # 排除文件夹本身和生成的 jsonl 数据集
        if os.path.isfile(file_path) and not filename.endswith('.jsonl'):
            new_path = os.path.join(raw_data_folder, filename)
            shutil.move(file_path, new_path)
            print(f"已移动: {filename} -> raw_materials/")

    files = os.listdir(raw_data_folder)
    if len(files) == 0:
        print("\n目前语料文件夹是空的，请上传您的文件到 raw_materials 文件夹。")
    else:
        print("\n当前 raw_materials 文件夹内有以下文件：")
        for f in files:
            print(f" - {f}")
else:
    print("❌ Google Drive 似乎还没挂载好。请确保您在左侧栏完成了授权。")

✅ 恭喜！检测到 Google Drive 已经挂载成功！

📁 主文件夹: /content/drive/MyDrive/Colab Notebooks/ow
📁 原始语料子文件夹已创建: /content/drive/MyDrive/Colab Notebooks/ow/raw_materials

整理文件中...
已移动: IDI notes transcirpt pack_Edrington_202607.docx -> raw_materials/
已移动: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI notes pack_Edrington_202607.docx -> raw_materials/
已移动: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI recruitment update.xlsx -> raw_materials/
已移动: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_酒Recruit Update.xlsx -> raw_materials/
已移动: Oliver Wyman_2026 Whisky 1-on-1 in-depth interview_v1_20260701.docx -> raw_materials/
已移动: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI note summary.pptx -> raw_materials/
已移动: 2026 OW APR Offsite AI Challenge Launch.pptx -> raw_materials/
已移动: IDI note summary v2.pptx -> raw_materials/

当前 raw_materials 文件夹内有以下文件：
 - IDI notes transcirpt pack_Edrington_202607.docx
 - cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_ID

### 4. 解析数据并生成 JSONL 训练集 (Parse Data to JSONL)

我们需要读取您上传的 PDF、Word 或 TXT 文件，并将其转换为大模型微调所需的 `JSONL` 格式。我们将文本切分成段落，并应用一个基础的“指令-回复”模板。

In [ ]:
# 安装读取 PDF 和 Word 文件的依赖库
!pip install -q PyPDF2 python-docx

print("文档解析库安装完成！")

文档解析库安装完成！


In [ ]:
import os
import json
import PyPDF2
import docx
import re

data_folder = '/content/drive/MyDrive/Colab Notebooks/ow'
raw_data_folder = os.path.join(data_folder, 'raw_materials')
output_jsonl = os.path.join(data_folder, 'training_data.jsonl')

def clean_text(text):
    """清洗文本，去除类似 'Speaker 1 29:45' 的无用标记"""
    # 去除 Speaker + 数字 + 时间戳 (例如 Speaker 1 29:45)
    text = re.sub(r'Speaker\s*\d+\s*\d+:\d+', '', text, flags=re.IGNORECASE)
    # 去除单独的发言人标记 (例如 Speaker 1: 或 面试官:)
    text = re.sub(r'(Speaker\s*\d+|Interviewer|Candidate|面试官|候选人|受访者)[:：\s]*', '', text, flags=re.IGNORECASE)
    # 去除单独的时间戳 (例如 29:45)
    text = re.sub(r'\b\d{1,2}:\d{2}\b', '', text)
    # 替换多个换行和空格
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_text(file_path):
    """根据文件后缀提取文本"""
    text = ""
    if file_path.lower().endswith('.pdf'):
        with open(file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                if page.extract_text():
                    text += page.extract_text() + "\n"
    elif file_path.lower().endswith('.docx'):
        doc = docx.Document(file_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
    elif file_path.lower().endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
    return clean_text(text)

print(f"开始处理 {raw_data_folder} 中的文件...")
dataset = []

# 遍历子文件夹内的所有原始文件
for filename in os.listdir(raw_data_folder):
    file_path = os.path.join(raw_data_folder, filename)
    if os.path.isfile(file_path):
        print(f"正在读取: {filename}")
        try:
            content = extract_text(file_path)
            if content.strip():
                # 将长文本按句子或换行简单切分（过滤掉过短的无意义段落）
                paragraphs = [p.strip() for p in content.split('。') if len(p.strip()) > 30]
                for para in paragraphs:
                    # 构建基础的训练格式 (Instruction, Input, Output)
                    entry = {
                        "instruction": "作为一名专业的咨询公司面试官，请根据以下知识或案例上下文进行专业的回复。",
                        "input": "",
                        "output": para + "。"
                    }
                    dataset.append(entry)
        except Exception as e:
            print(f"读取 {filename} 时出错: {e}")

# 保存为 JSONL 文件 (保存在主目录)
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"\n✅ 数据清洗与处理完成！共生成了 {len(dataset)} 条训练数据段落。")
print(f"训练文件已保存至: {output_jsonl}")

开始处理 /content/drive/MyDrive/Colab Notebooks/ow/raw_materials 中的文件...
正在读取: IDI notes transcirpt pack_Edrington_202607.docx
正在读取: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI notes pack_Edrington_202607.docx
正在读取: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI recruitment update.xlsx
正在读取: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_酒Recruit Update.xlsx
正在读取: Oliver Wyman_2026 Whisky 1-on-1 in-depth interview_v1_20260701.docx
正在读取: cce0d0ed9cc2be6e95652125a020799d_2795725677132292924_m_IDI note summary.pptx
正在读取: 2026 OW APR Offsite AI Challenge Launch.pptx
正在读取: IDI note summary v2.pptx

✅ 数据清洗与处理完成！共生成了 1467 条训练数据段落。
训练文件已保存至: /content/drive/MyDrive/Colab Notebooks/ow/training_data.jsonl


### 5. 加载模型与微调 (Load Model and Fine-tune with Unsloth)

我们将加载 `Qwen2-7B-Instruct` 的 4-bit 量化版本，这极大节省了显存。同时加载 LoRA 适配器来进行微调。

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # 最大上下文长度降至 1024 以提速
dtype = None # 自动检测数据类型
load_in_4bit = True # 使用 4bit 量化以减少内存使用

# 1. 加载模型和 Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-7B-Instruct-bnb-4bit", # 适合中英双语的优秀开源模型
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. 添加 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA 秩，建议值为 8, 16, 32, 64
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 优化最快
    bias = "none",    # None 优化最快
    use_gradient_checkpointing = "unsloth", # 减少显存占用
    random_state = 3407,
    use_rslora = False,  # 是否使用 Rank Stabilized LoRA
    loftq_config = None, # LoftQ
)

print("\n✅ 模型和 LoRA 适配器加载成功！最大序列长度已调整为 1024。")


==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


✅ 模型和 LoRA 适配器加载成功！最大序列长度已调整为 1024。


In [ ]:
from datasets import load_dataset

# 读取刚刚使用大模型重写的高质量数据集！
data_path = '/content/drive/MyDrive/Colab Notebooks/ow/training_data_llm_rewritten.jsonl'
dataset = load_dataset("json", data_files=data_path, split="train")

# 构建提示词模板
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # 将指令、输入和输出格式化，并加上 EOS_TOKEN 结束符
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

formatted_dataset = dataset.map(formatting_prompts_func, batched = True)
print("\n✅ 高质量数据集格式化完毕，准备开始全新训练！")


✅ 高质量数据集格式化完毕，准备开始全新训练！


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 设置训练器
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # 提升至 4 以加速
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # 降至 300 步以加速
        # num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # 禁用自动保存以避免 SFTConfig PicklingError
    ),
)

# 开始训练
print("开始训练...")
trainer_stats = trainer.train()
print("\n🎉 训练完成！")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


开始训练...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,460 | Num Epochs = 4 | Total steps = 300
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
10,2.325254
20,1.598306
30,1.432135
40,1.424506
50,1.363139
60,1.345127
70,1.347167
80,1.290517
90,1.306556
100,1.211628



🎉 训练完成！


### 6. 测试微调后的模型 (Test the Fine-tuned Model)

让我们来看看您的专属“咨询面试官”表现如何！我们将输入一个测试问题，看看它生成的回复。

In [ ]:
# 启用 Unsloth 的原生 2 倍速推理模式
FastLanguageModel.for_inference(model)

# 准备一个测试问题
test_instruction = "作为一名专业的咨询公司面试官，请根据以下知识或案例上下文进行专业的回复。"
test_input = "候选人你好，我们现在开始 Case Interview。假设现在有一家欧洲的高端烈酒品牌想进入中国下沉市场，请问你会如何分析这个 Market Entry 的机会？"

test_prompt = alpaca_prompt.format(
    test_instruction,
    test_input,
    "", # output 留空，等待模型生成
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

print("🤔 面试官正在思考中...\n")
# 生成回复 (max_new_tokens 控制回答的最大长度)
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

# 提取并打印出最终的回复内容
final_answer = response.split("### Response:\n")[-1]
print(f"🗣️ 面试官回复：\n{final_answer}")

🤔 面试官正在思考中...



Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🗣️ 面试官回复：
这是一个典型的 Market Entry 案例，我建议采用一个结构化的框架来拆解：第一，Market Attractiveness，我们需要评估下沉市场的容量、增长趋势以及竞争格局；第二，Competitive Landscape，分析现有玩家的定位和壁垒；第三，Customer Segmentation，识别目标客群及其消费场景；第四，Channel Strategy，考虑分销模式是直营还是通过本地经销商；第五，Pricing Strategy，定价是否需要采用渗透定价还是溢价策略；第六，Supply Chain & Logistics，评估物流成本和库存周转。我会先从第一和第二点切入，初步判断这个机会是否值得进一步投入资源。


### 7. 保存模型权重 (Save the Model to Google Drive)

将训练好的 LoRA 权重保存到您的云盘中，这样下次直接加载即可，无需重新等待训练。

In [ ]:
import os

# 定义保存路径
save_path = "/content/drive/MyDrive/Colab Notebooks/ow/lora_model"

# 保存模型和 Tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n✅ 恭喜！模型微调权重已成功保存到您的云盘：{save_path}")
print("下次使用时，您可以直接加载这个微调过的权重，无需重新训练！")


✅ 恭喜！模型微调权重已成功保存到您的云盘：/content/drive/MyDrive/Colab Notebooks/ow/lora_model
下次使用时，您可以直接加载这个微调过的权重，无需重新训练！


### 8. 进阶路线 (Route 2)：使用 LLM API 重写高质量数据集 (Data Distillation)

如果基础模型的效果不够“聪明”，我们可以使用一个强大的大模型（如 GPT-4, Claude, DeepSeek, Qwen 等）将原始会议记录改写为完美的“面试官 QA 问答对”。

In [ ]:
# 安装通用大模型 API 客户端库和进度条库
!pip install -q openai tqdm

In [ ]:
# @title ⚙️ 大模型 API 配置与模型选择 (合并简化版)
# @markdown 请在右侧填写您的 API 信息（已为您修正了之前填反的链接和密钥）。运行此单元格后，会直接弹出模型选择框。

api_key = "sk-ws-H.ELLRYDD.6PPk.MEYCIQDizREa3CLBdxMfATh8lMLW8r1468-4GZf-4HtVCGnwpgIhAMC4XsmA0kTp0SSGLZ5iFozZGsXz3pU2hzQ8SzI4ZGSm" # @param {type:"string"}
base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1" # @param {type:"string"}

import os
import requests
import ipywidgets as widgets
from IPython.display import display

os.environ["CUSTOM_API_KEY"] = api_key
os.environ["CUSTOM_BASE_URL"] = base_url

print("正在连接 API 获取可用模型列表...")
headers = {"Authorization": f"Bearer {api_key}"}

try:
    # 请求标准 OpenAI 格式的 models 端点
    response = requests.get(f"{base_url}/models", headers=headers, timeout=10)
    response.raise_for_status()
    models_data = response.json()

    # 解析出模型 ID 列表并排序
    model_ids = [m["id"] for m in models_data.get("data", [])]
    model_ids.sort()

    if not model_ids:
        print("⚠️ API 返回成功，但列表为空。")
    else:
        print(f"✅ 成功获取到 {len(model_ids)} 个模型！")

        # 创建交互式下拉菜单
        dropdown = widgets.Dropdown(
            options=model_ids,
            value=model_ids[0],
            description='🤖 选择模型:',
            disabled=False,
        )

        # 设置默认值
        os.environ["CUSTOM_MODEL_NAME"] = model_ids[0]

        # 监听下拉菜单的变化
        def on_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                os.environ["CUSTOM_MODEL_NAME"] = change['new']
                print(f"\n👉 目标模型已切换为: {change['new']}")

        dropdown.observe(on_change, names='value')
        display(dropdown)
        print(f"当前默认使用: {model_ids[0]} (如需更改请在上方下拉框直接选择)")
        print("✨ 选择完成后，您不需要再运行任何配置，直接向下运行带进度条的重写代码即可！")

except Exception as e:
    print(f"❌ 获取模型列表失败: {e}")
    print("请检查您的 Base URL 或 API Key 是否有效。")

正在连接 API 获取可用模型列表...
✅ 成功获取到 234 个模型！


Dropdown(description='🤖 选择模型:', options=('MiniMax-M2.1', 'MiniMax-M2.5', 'MiniMax/MiniMax-M2.1', 'MiniMax/Mini…

当前默认使用: MiniMax-M2.1 (如需更改请在上方下拉框直接选择)
✨ 选择完成后，您不需要再运行任何配置，直接向下运行带进度条的重写代码即可！


In [ ]:
import os
import json
import concurrent.futures
from openai import OpenAI
from tqdm import tqdm

# 1. 初始化客户端
client = OpenAI(
    api_key=os.environ.get("CUSTOM_API_KEY"),
    base_url=os.environ.get("CUSTOM_BASE_URL")
)
model_name = os.environ.get("CUSTOM_MODEL_NAME")

input_file = '/content/drive/MyDrive/Colab Notebooks/ow/training_data.jsonl'
output_file = '/content/drive/MyDrive/Colab Notebooks/ow/training_data_llm_rewritten.jsonl'

# 2. 读取我们之前物理清洗过的数据
raw_data = []
if os.path.exists(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            raw_data.append(json.loads(line))

print(f"已加载 {len(raw_data)} 条原始数据。")

# 系统提示词：定义大模型作为“数据标注员”的角色
system_prompt = """
你是一个顶级的数据清洗和构建专家。你的任务是将用户提供的“松散的、口语化的访谈记录”重写为适用于训练大模型的“高质量面试问答对”。
请输出一段候选人的可能回答（作为 Input），以及一段专业咨询面试官的追问或评价（作为 Output）。
要求：
1. 剔除所有类似“Speaker 1”、“恩”、“啊”等口语化废话。
2. 确保 Output 具有极强的逻辑性和专业咨询感（例如提到 Framework、Market Entry、定价策略等）。
3. 必须以严格的 JSON 格式输出，包含 `input` 和 `output` 两个字段，不要输出任何额外的说明文本。
"""

# 定义单条处理函数
def process_item(item_tuple):
    i, item = item_tuple
    raw_text = item["output"]
    user_prompt = f"请重写以下访谈片段：\n{raw_text}"

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" }, # 强制输出 JSON
            temperature=0.3
        )

        result_str = response.choices[0].message.content
        result_json = json.loads(result_str)

        # 兼容模型偶尔返回 JSON 数组的情况
        if isinstance(result_json, list) and len(result_json) > 0:
            result_json = result_json[0]

        # 组装新的高质量数据
        return {
            "status": "success",
            "data": {
                "instruction": "作为一名专业的咨询公司面试官，请根据候选人的回答进行专业的回复与追问。",
                "input": result_json.get("input", ""),
                "output": result_json.get("output", "")
            }
        }
    except Exception as e:
        return {"status": "error", "msg": f"第 {i} 条处理出错: {e}"}

# 3. 开始多线程重写
rewritten_dataset = []
test_limit = len(raw_data) # 处理全部数据
max_workers = 80 # 💥 进一步拉高到 80 并发

print(f"\n🚀 开始调用 {model_name} 进行 [极速多线程] 重写 (共 {test_limit} 条，并发数: {max_workers})...")

# 使用 ThreadPoolExecutor 进行并发请求
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    # 提交所有任务
    futures = [executor.submit(process_item, (i, item)) for i, item in enumerate(raw_data[:test_limit])]

    # 使用 tqdm 显示进度条
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        result = future.result()
        if result["status"] == "success":
            rewritten_dataset.append(result["data"])
        else:
            # 打印错误信息但不中断整体流程
            tqdm.write(result["msg"])

# 4. 保存高质量数据集
with open(output_file, 'w', encoding='utf-8') as f:
    for item in rewritten_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"\n✅ 大模型 [极速多线程] 重写完成！成功生成了 {len(rewritten_dataset)} 条高质量数据。")
print(f"保存至: {output_file}")
print("\n下一步：请向上滚动重新运行第 5 步的数据格式化单元格，并开始重新训练！")

已加载 1467 条原始数据。

🚀 开始调用 qwen3.7-plus 进行 [极速多线程] 重写 (共 1467 条，并发数: 50)...


  3%|▎         | 40/1467 [00:31<09:18,  2.56it/s]

第 18 条处理出错: 'list' object has no attribute 'get'


 23%|██▎       | 331/1467 [03:06<12:34,  1.51it/s]

第 334 条处理出错: 'list' object has no attribute 'get'


...

第 804 条处理出错: 'list' object has no attribute 'get'


 63%|██████▎   | 924/1467 [08:12<04:23,  2.06it/s]

第 895 条处理出错: 'list' object has no attribute 'get'


100%|██████████| 1467/1467 [13:33<00:00,  1.80it/s]

第 1417 条处理出错: 'list' object has no attribute 'get'

✅ 大模型 [极速多线程] 重写完成！成功生成了 1460 条高质量数据。
保存至: /content/drive/MyDrive/Colab Notebooks/ow/training_data_llm_rewritten.jsonl

下一步：请向上滚动重新运行第 5 步的数据格式化单元格，并开始重新训练！


# Task
Stop the current training, restart the runtime to clear GPU memory, and modify the fine-tuning parameters to speed up the process.

## 准备加速环境

### Subtask:
指导用户中断训练并重启会话


### 🛑 步骤 1：准备加速环境

为了能够极速重启并释放全部 GPU 显存，请您按照以下步骤操作：

1. **中断当前训练**：如果上方的微调训练代码（单元格5）仍在运行，请点击该单元格左侧的“停止”按钮（方形图标）。
2. **重启运行时**：点击顶部菜单栏的 **Runtime (代码执行程序)** -> **Restart session (重启会话)**。这会清空之前的内存和显存，确保新的训练不会报错 OOM (Out Of Memory)。

完成后，我们将修改参数以加快下一步的训练。

## 修改训练代码以提速

### Subtask:
Modify the fine-tuning parameters in the notebook to speed up the training process.


### 9. 进阶微调 (Advanced Fine-Tuning with High-Quality Data)

使用第 8 步中大模型重写的高质量问答对，对现有的模型进行第二阶段的进阶微调，让 AI 面试官的回复逻辑更清晰、更符合顶级咨询公司的标准。

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os

# 1. 读取刚才生成的高质量数据集
data_path = '/content/drive/MyDrive/Colab Notebooks/ow/training_data_llm_rewritten.jsonl'

if not os.path.exists(data_path):
    print("❌ 找不到高质量数据集，请确认第 8 步已成功生成文件。")
else:
    dataset_v2 = load_dataset("json", data_files=data_path, split="train")

    # 2. 重新格式化数据
    formatted_dataset_v2 = dataset_v2.map(formatting_prompts_func, batched = True)
    print(f"\n✅ 成功加载并格式化了 {len(formatted_dataset_v2)} 条高质量重写数据！")

    # 3. 配置新一轮的训练器 (Trainer)
    trainer_v2 = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = formatted_dataset_v2,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 4,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            max_steps = 300, # 您可以根据需要调整，建议 300-500 步
            learning_rate = 2e-4,
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs_v2",
            save_strategy = "no",
        ),
    )

    # 4. 开始第二阶段进阶训练
    print("\n🚀 开始基于高质量数据进行第二阶段微调...")
    trainer_v2_stats = trainer_v2.train()
    print("\n🎉 第二阶段微调完成！您的 AI 面试官现在具备了更强的专业推理能力。")


✅ 成功加载并格式化了 1460 条高质量重写数据！


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/1460 [00:00<?, ? examples/s]


🚀 开始基于高质量数据进行第二阶段微调...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,460 | Num Epochs = 4 | Total steps = 300
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.956004
20,0.949090
30,0.926121
40,0.954663
50,0.928061
60,0.923574
70,0.943576
80,0.908054
90,0.928918
100,0.707090


Step,Training Loss
10,0.956004
20,0.949090
30,0.926121
40,0.954663
50,0.928061
60,0.923574
70,0.943576
80,0.908054
90,0.928918
100,0.707090



🎉 第二阶段微调完成！您的 AI 面试官现在具备了更强的专业推理能力。


### 10. 测试进阶模型并保存
进阶训练完成后，我们可以用同一个问题再次测试它，对比一下效果，并覆盖保存最新的强力权重。

In [ ]:
# 启用快速推理模式
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "作为一名专业的咨询公司面试官，请根据以下知识或案例上下文进行专业的回复。",
    "候选人你好，我们现在开始 Case Interview。假设现在有一家欧洲的高端烈酒品牌想进入中国下沉市场，请问你会如何分析这个 Market Entry 的机会？",
    "", # 留空
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

print("🤔 进阶版面试官正在思考中...\n")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

final_answer = response.split("### Response:\n")[-1]
print(f"🗣️ 进阶版面试官回复：\n{final_answer}")

# 覆盖保存为最新的强大模型
save_path = "/content/drive/MyDrive/Colab Notebooks/ow/lora_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n✅ 进阶版模型权重已覆盖保存至：{save_path}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤔 进阶版面试官正在思考中...

🗣️ 进阶版面试官回复：
这是一个典型的 Market Entry 案例，我建议用 MECE 原则构建一个系统性的分析框架。首先从宏观层面（Macro）审视 China 市场的宏观环境，包括消费结构的变迁、新一线/下沉市场的渠道特征以及竞争格局；其次深入到行业层面（Industry），评估烈酒品类在本土市场的接受度、定价策略的可行性以及品牌本地化的潜力；最后聚焦于客户自身的 Portfolio 结构——如果该品牌目前高度依赖欧洲高净值人群，其产品线是否具备适配中国年轻化、二三线城市消费升级趋势的能力？我建议先用 2-8 法则验证 80% 的增长机会究竟来源于哪 20% 的细分场景，再决定是自建渠道还是通过现有经销商网络进行轻资产扩张。

✅ 进阶版模型权重已覆盖保存至：/content/drive/MyDrive/Colab Notebooks/ow/lora_model


### 11. 测试消费者深度访谈场景 (Test Consumer IDI Scenario)

模拟消费者调研场景，测试 AI 是否能针对受访者的回答进行有商业洞察价值的追问。

In [ ]:
# 启用快速推理模式 (如果尚未启用)
FastLanguageModel.for_inference(model)

# 设定符合您真实语料场景的指令
idi_instruction = "作为一名专业的咨询公司调研访问者，你正在与一位潜在的烈酒消费者进行深度访谈。请根据受访者的回答进行专业的回复与深度追问，以挖掘市场洞察。"

# 模拟一位消费者的回答
idi_input = "我平时主要是在商务宴请的时候会喝高端烈酒，比如轩尼诗或者麦卡伦。自己在家其实很少喝，总觉得气氛不太对。不过最近被朋友安利，尝试了一些日本威士忌，感觉口感偏甜，还挺好入口的。"

test_prompt_idi = alpaca_prompt.format(
    idi_instruction,
    idi_input,
    "", # 留空，等待模型生成追问
)

inputs_idi = tokenizer([test_prompt_idi], return_tensors = "pt").to("cuda")

print("🤔 调研面试官正在思考如何追问...\n")
outputs_idi = model.generate(**inputs_idi, max_new_tokens = 512, use_cache = True)
response_idi = tokenizer.batch_decode(outputs_idi, skip_special_tokens = True)[0]

final_answer_idi = response_idi.split("### Response:\n")[-1]
print(f"🗣️ AI 调研官回复/追问：\n{final_answer_idi}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤔 调研面试官正在思考如何追问...

🗣️ AI 调研官回复/追问：
感谢您的分享。从您的回答中，我们可以提炼出几个关键洞察：首先，高端烈酒的消费场景高度依赖社交属性，如商务宴请等正式场合；其次，家庭自饮场景存在明显的饮用门槛，可能与氛围、文化或习惯有关。您提到最近尝试日本威士忌并接受了其偏甜的口感，这暗示了产品在口感创新方面有一定的市场潜力。接下来，我想进一步探讨：在您看来，日本威士忌相较于传统苏格兰威士忌，在口感和消费体验上有哪些核心差异？这些差异是否足以影响您从传统品牌转向新品牌的选择？


### 12. 接入语音模块 (Voice Integration - Text to Speech)

现在我们已经拥有了高质量的文本回复。接下来，我们将使用 `edge-tts` (免费且效果出色的文本转语音库)，为您的 AI 面试官赋予一个专业的声音，并直接在笔记本中播放。

In [ ]:
!pip install -q edge-tts

import edge_tts
import asyncio
from IPython.display import Audio, display

# 获取我们在第 11 步中生成的回复文本，如果因为重启丢失了变量，则使用备用的测试文本
try:
    text_to_speak = final_answer_idi
except NameError:
    text_to_speak = "感谢您的分享。从您的回答中，我们可以提炼出几个关键洞察：首先，高端烈酒的消费场景高度依赖社交属性；其次，家庭自饮场景存在明显的饮用门槛。接下来，我想进一步探讨：在您看来，日本威士忌相较于传统苏格兰威士忌，在口感和消费体验上有哪些核心差异？"

output_audio_file = "ai_interviewer_response.mp3"

# 选择一个专业的中文发音人
# 备选: 'zh-CN-XiaoxiaoNeural' (女性), 'zh-CN-YunxiNeural' (年轻男性)
voice = "zh-CN-YunxiNeural"

async def generate_audio():
    print(f"🎙️ 正在将 AI 的回复转化为语音 (使用音色: {voice})...")
    communicate = edge_tts.Communicate(text_to_speak, voice)
    await communicate.save(output_audio_file)
    print("✅ 语音生成完毕！请点击下方的播放按钮收听：")

# 运行异步生成任务
await generate_audio()

# 在 Colab 中显示音频播放器
display(Audio(output_audio_file, autoplay=True))

🎙️ 正在将 AI 的回复转化为语音 (使用音色: zh-CN-YunxiNeural)...
✅ 语音生成完毕！请点击下方的播放按钮收听：


### Connecting front and backend into one complete platform

In [ ]:
!pip install -q openai-whisper
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q snownlp

import asyncio
import edge_tts
import whisper
import torch
import numpy as np
from IPython.display import Audio, display
from unsloth import FastLanguageModel
import os
from snownlp import SnowNLP

# 1. 加载微调后的模型和 Tokenizer
save_path = "/content/drive/MyDrive/Colab Notebooks/ow/lora_model"

if 'model' not in locals() or 'tokenizer' not in locals():
    print("正在加载微调后的模型...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = save_path,
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)
    print("模型加载完成并进入推理模式。")

# 加载 Whisper 模型用于语音转文本 (ASR)
print("正在加载 Whisper ASR 模型...")
whisper_model = whisper.load_model("base")
print("Whisper 模型加载完成。")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token

async def conversational_loop():
    print("\n===============================")
    print("--- 🌟 智能咨询调研系统已启动 ---")
    print("===============================")

    interview_topic = input("\n📝 请设定本次调研的主题或客户背景 (例如: 威士忌下沉市场调研): ")
    if not interview_topic.strip():
        interview_topic = "通用商业咨询"

    print(f"\n✅ 已锁定调研主题: 【{interview_topic}】")
    print("请开始您的提问，或说 '再见' 结束对话。")

    while True:
        user_audio_input_text = input("\n您对面试官说: ")

        if user_audio_input_text.lower() == '再见':
            print("面试官: 期待与您的下次交流！")
            break

        user_query_text = user_audio_input_text
        print(f"[ASR 结果]: {user_query_text}")

        # 增加: 情感分析模块
        try:
            s = SnowNLP(user_query_text)
            sentiment_score = s.sentiments # 0.0 (消极) 到 1.0 (积极)
            if sentiment_score > 0.6:
                sentiment_label = "积极 (Positive)"
            elif sentiment_score < 0.4:
                sentiment_label = "消极 (Negative)"
            else:
                sentiment_label = "中性 (Neutral)"
            print(f"[📊 情感分析]: 得分 {sentiment_score:.2f} - 当前情绪判定为【{sentiment_label}】")
        except:
            sentiment_label = "未知"

        # 3. LLM 处理并生成回复 (动态插入主题和候选人情绪)
        instruction = f"作为一名专业的咨询公司面试官和调研员，本次访谈的核心主题是：【{interview_topic}】。注意，候选人当前的回答情绪为：【{sentiment_label}】。请紧扣主题并顾及候选人情绪，进行高度专业的回复与深度追问，挖掘深刻的市场洞察。"
        input_text = user_query_text

        full_prompt = alpaca_prompt.format(
            instruction,
            input_text,
            "", # output 留空
        )

        inputs = tokenizer([full_prompt], return_tensors = "pt").to("cuda")

        print("🤔 面试官正在思考中...")
        outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
        response_text = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

        ai_response_pure_text = response_text.split("### Response:\n")[-1].strip()
        print(f"🗣️ 面试官回复: {ai_response_pure_text}")

        # 4. TTS 合成语音
        output_audio_file = "ai_interviewer_response.mp3"
        voice = "zh-CN-YunxiNeural"

        print(f"🎙️ 正在将回复转化为语音...")
        communicate = edge_tts.Communicate(ai_response_pure_text, voice)
        await communicate.save(output_audio_file)

        # 5. 播放
        display(Audio(output_audio_file, autoplay=True))

# 运行对话循环
await conversational_loop()


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-1irukqgd/unsloth_cac001c0b3f74ee68bccccec16df42b9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-1irukqgd/unsloth_cac001c0b3f74ee68bccccec16df42b9
  Resolved https://github.com/unslothai/unsloth.git to commit a43c81d529fc446fca3256ab7f73793f5e201ca5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 19.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
正在加载微调后的模型...
==((====))==  Unsloth 2026.8.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA 

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


模型加载完成并进入推理模式。
正在加载 Whisper ASR 模型...
Whisper 模型加载完成。

--- 🌟 智能咨询调研系统已启动 ---

📝 请设定本次调研的主题或客户背景 (例如: 威士忌下沉市场调研): 香水下沉市场调研

✅ 已锁定调研主题: 【香水下沉市场调研】
请开始您的提问，或说 '再见' 结束对话。

您对面试官说: 你好
[ASR 结果]: 你好
[📊 情感分析]: 得分 0.53 - 当前情绪判定为【中性 (Neutral)】
🤔 面试官正在思考中...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ASR 结果]: sometimes 
[📊 情感分析]: 得分 0.53 - 当前情绪判定为【中性 (Neutral)】
🤔 面试官正在思考中...
🗣️ 面试官回复: 感谢您的分享。为了更系统地梳理我们的讨论，让我们回到本次调研的核心目标：深入理解香水品类在下沉市场的消费者行为与需求痛点。您刚才提到的‘有时’场景，能否具体展开说明？例如，在什么特定情境下、针对哪些人群画像，以及伴随哪些消费动机下，您或身边的用户会选择购买或使用香水产品？这样的场景细分将帮助我们构建更精准的目标客群画像，并驱动后续的产品定位与定价策略分析。
🎙️ 正在将回复转化为语音...



您对面试官说: 出门前


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ASR 结果]: 出门前
[📊 情感分析]: 得分 0.77 - 当前情绪判定为【积极 (Positive)】
🤔 面试官正在思考中...
🗣️ 面试官回复: 感谢您的分享。从您的回答中，我注意到您提到了‘出门前’这一场景。为了更好地理解这一情境，我想请您进一步思考：在‘出门前’这个时间节点上，消费者的核心决策驱动因素是什么？例如，是受到特定场合（如约会、聚会）的驱动，还是基于即时的情绪需求？此外，您认为‘出门前’这一场景与其他消费决策节点（如购买后或睡前）在消费者心理图谱中占据怎样的位置？这将有助于我们构建更精准的用户旅程地图 (User Journey Map)。
🎙️ 正在将回复转化为语音...


In [ ]:
import os
from IPython.display import HTML, display, Audio
from google.colab.output import eval_js
from base64 import b64decode

# --- 1. 定义网页端录音的 UI 和逻辑 (JavaScript) ---
RECORD_JS = """
<script>
  var my_btn = document.createElement("BUTTON");
  my_btn.innerHTML = "🎙️ 点击开始录音";
  my_btn.style.fontSize = "16px";
  my_btn.style.padding = "10px";
  my_btn.style.margin = "10px 0";
  my_btn.style.borderRadius = "8px";
  my_btn.style.cursor = "pointer";
  my_btn.style.backgroundColor = "#4CAF50";
  my_btn.style.color = "white";
  my_btn.style.border = "none";
  document.body.appendChild(my_btn);

  var base64data = "";
  var recorder, gumStream;

  var dataPromise = new Promise(resolve => {
    my_btn.onclick = async () => {
      if (my_btn.innerHTML === "🎙️ 点击开始录音") {
        my_btn.innerHTML = "⏹️ 录音中... (点击停止)";
        my_btn.style.backgroundColor = "#f44336";
        // 请求麦克风权限
        const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        gumStream = stream;
        recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.onstop = async () => {
          const blob = new Blob(chunks, { type: 'audio/webm' });
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          reader.onloadend = () => {
            base64data = reader.result;
            resolve(base64data);
          }
        };
        recorder.start();
      } else {
        my_btn.innerHTML = "⏳ 处理中...";
        my_btn.style.backgroundColor = "#9e9e9e";
        recorder.stop();
        gumStream.getAudioTracks()[0].stop();
      }
    };
  });
</script>
"""

def get_audio_from_mic():
    """在 Colab 前端显示录音按钮，并将录制的音频保存为 wav 文件"""
    display(HTML(RECORD_JS))
    # 等待前端 JS promise resolve 返回 base64 数据
    data = eval_js("dataPromise")
    binary = b64decode(data.split(',')[1])

    # 临时保存为 webm
    with open("user_mic.webm", "wb") as f:
        f.write(binary)

    # 使用 ffmpeg 转码为 16kHz 的单声道 wav (Whisper 的推荐格式)
    os.system("ffmpeg -y -i user_mic.webm -ar 16000 -ac 1 user_mic.wav >/dev/null 2>&1")
    return "user_mic.wav"

# --- 2. 结合全链路的语音对话循环 ---
async def conversational_loop_with_voice():
    print("\n=============================================")
    print("--- 🌟 智能咨询调研系统 (全语音交互版) 已启动 ---")
    print("=============================================")

    interview_topic = input("\n📝 请使用键盘设定本次调研的主题或客户背景: ")
    if not interview_topic.strip():
        interview_topic = "通用商业咨询"

    print(f"\n✅ 已锁定调研主题: 【{interview_topic}】")
    print("💡 提示：如果想结束对话，请在录音时说出 '再见' 或 '结束'。\n")

    while True:
        print("\n🎤 请点击下方绿色按钮开始说话...")
        # 1. 获取麦克风录音
        audio_file = get_audio_from_mic()

        print("🔄 正在转录您的语音...")
        # 2. Whisper ASR 语音转文本
        result = whisper_model.transcribe(audio_file, language="zh")
        user_query_text = result["text"].strip()

        print(f"\n[🗣️ 您的语音识别结果]: {user_query_text}")

        if not user_query_text:
            print("⚠️ 未识别到清晰的语音，请重试。")
            continue

        if "再见" in user_query_text or "结束" in user_query_text:
            print("🤖 面试官: 期待与您的下次交流！再见！")
            break

        # 3. 情感分析 (SnowNLP)
        try:
            s = SnowNLP(user_query_text)
            sentiment_score = s.sentiments
            if sentiment_score > 0.6:
                sentiment_label = "积极 (Positive)"
            elif sentiment_score < 0.4:
                sentiment_label = "消极 (Negative)"
            else:
                sentiment_label = "中性 (Neutral)"
            print(f"[📊 情感分析]: 得分 {sentiment_score:.2f} - 【{sentiment_label}】")
        except:
            sentiment_label = "未知"

        # 4. LLM 大模型推理
        instruction = f"作为一名专业的咨询公司面试官和调研员，本次访谈的主题是：【{interview_topic}】。注意候选人情绪：【{sentiment_label}】。请紧扣主题并顾及情绪，进行专业回复与追问。"
        full_prompt = alpaca_prompt.format(instruction, user_query_text, "")
        inputs = tokenizer([full_prompt], return_tensors = "pt").to("cuda")

        print("🤔 面试官正在思考中...")
        outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
        response_text = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
        ai_response_pure_text = response_text.split("### Response:\n")[-1].strip()

        print(f"\n🤖 面试官回复: {ai_response_pure_text}")

        # 5. TTS 文本转语音并播放
        output_audio_file = "ai_interviewer_response.mp3"
        print(f"🎙️ 正在将回复转化为语音...")
        communicate = edge_tts.Communicate(ai_response_pure_text, "zh-CN-YunxiNeural")
        await communicate.save(output_audio_file)

        display(Audio(output_audio_file, autoplay=True))

# 运行全语音版循环
await conversational_loop_with_voice()



--- 🌟 智能咨询调研系统 (全语音交互版) 已启动 ---

📝 请使用键盘设定本次调研的主题或客户背景: 运动饮料客户

✅ 已锁定调研主题: 【运动饮料客户】
💡 提示：如果想结束对话，请在录音时说出 '再见' 或 '结束'。


🎤 请点击下方绿色按钮开始说话...


### 13. 构建完整 Web UI (Gradio User Interface)

Let's wrap the entire pipeline into a beautiful web interface using Gradio. This will give you a proper chat window, a native microphone recording button, and audio playback.

In [ ]:
!pip install -q gradio

import gradio as gr
import edge_tts
import asyncio
import os
from snownlp import SnowNLP

# Main processing function for the UI
async def process_voice_chat(audio_file, history, topic):
    if not audio_file:
        return history, None

    # 1. Transcribe with Whisper
    result = whisper_model.transcribe(audio_file, language="zh")
    user_text = result["text"].strip()

    if not user_text:
        return history, None

    # 2. Sentiment Analysis
    try:
        s = SnowNLP(user_text)
        sentiment_score = s.sentiments
        if sentiment_score > 0.6:
            sentiment_label = "积极 (Positive)"
        elif sentiment_score < 0.4:
            sentiment_label = "消极 (Negative)"
        else:
            sentiment_label = "中性 (Neutral)"
    except:
        sentiment_label = "未知"

    # 3. LLM Inference
    instruction = f"作为一名专业的咨询公司面试官和调研员，本次访谈的主题是：【{topic}】。注意候选人情绪：【{sentiment_label}】。请紧扣主题并顾及情绪，进行专业回复与追问。"
    full_prompt = alpaca_prompt.format(instruction, user_text, "")
    inputs = tokenizer([full_prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    response_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    ai_text = response_text.split("### Response:\n")[-1].strip()

    # 4. TTS Generation
    output_audio = "gradio_response.mp3"
    communicate = edge_tts.Communicate(ai_text, "zh-CN-YunxiNeural")
    await communicate.save(output_audio)

    # Update chat history
    history.append((user_text, ai_text))

    return history, output_audio

# Build the Gradio UI
with gr.Blocks(title="AI 咨询面试官", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌟 智能咨询调研系统 (Web UI 版)")
    gr.Markdown("设定好主题后，点击下方麦克风录音，AI 面试官会自动回复并播放语音。")

    with gr.Row():
        topic_input = gr.Textbox(label="📝 设定调研主题", placeholder="例如: 威士忌下沉市场调研", value="通用商业咨询")

    chatbot = gr.Chatbot(label="💬 对话历史", height=400)

    with gr.Row():
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="🎤 点击录音进行回答")
        audio_out = gr.Audio(label="🗣️ AI 面试官语音回复", autoplay=True)

    # Trigger the process when the user stops recording
    audio_in.stop_recording(
        fn=process_voice_chat,
        inputs=[audio_in, chatbot, topic_input],
        outputs=[chatbot, audio_out]
    )

    # Also allow clearing the history
    clear_btn = gr.Button("🗑️ 清空对话记录")
    clear_btn.click(lambda: ([], None), None, [chatbot, audio_out])

# Launch the UI (creates a public shareable link)
demo.launch(debug=True, share=True)

/tmp/ipykernel_16658/2208370547.py:54: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI 咨询面试官", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://66208d5b3eb29711a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
